# Why read-only Python cells ask: the recorded cells, re-judged inside the dialect

The question (ruled by Shane, 2026-11-19, through the coordinator): of the Python cells a run tries inside the read-only
dialect, what share finishes inside; for the refused ones, why; how many approval cards that leaves per run; and how many
refused cells would finish inside had mutation of plain data the cell did not make (a full-path cell's, or the injected
`df`/`tables`) been allowed. It decides whether a design that lets the model edit its own kernel values unasked is worth
building.

No recorded run ever tried a cell inside: every bench and observe run is page-built, and page-hosted Python skips the
dialect (`sw-run-host.ts`, "this run's Python runs on the page, where nothing is tried inside the dialect"). So the
recorded cells are REPLAYED here, offline, through the fork core's inside try. That gives an answer under these caveats,
which bound everything below:

1. **One task.** Every cell is from `csv-total` ("what is the exact total of the `amount` column of /sales.csv"), in two
   sweeps of the prompt-budget study (`prompt-budget-cuts`, `-rep2`, 2026-11-13). A summing task is close to the best case
   for a read-only dialect; nothing here says how a task that reshapes data fares.
2. **The models believed Python ran on the page.** Their prompts and results were those of page-hosted Python, with no
   dialect to stay inside and every cell approved. A model told about the dialect, or shown its refusals, would write
   other cells.
3. **Six models, unevenly.** qwen3.6:35b wrote nearly half the cells. The pooled share is that model's; read the
   by-model table.

And three of the replay's own: the injected table is rebuilt as the worker's fetch cache holds it (`kind: "rows"`, from
the fixture's deterministic CSV), not from the run's bytes; the worker's lockdown is not installed (the wrapper runs
with `hardened` off, as `tests/python-dialect-inside.test.mjs` drives it); and a refused cell's full-path rerun has no
`ml` and no network, only a read of the task's /sales.csv served from the fixture, so a cell that needed more than that
leaves less behind for the cells after it.

In [1]:
import json, re
import pandas as pd
import wmlnb

HERE = wmlnb.ROOT / "notebooks" / "python-dialect"
PLAIN = HERE / "data" / "cell-replay" / "replay.json"
OWNED = HERE / "data" / "cell-replay" / "replay-owned-plain.json"
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 80)

def replay(mode, out):
    raise RuntimeError(f"re-extract with: node --import tsx notebooks/python-dialect/cell-replay/replay.mjs {mode} {out} <run.json>... "
                       "(the 58 exports are listed by the extract's source; dist/pyodide must be the fork core)")

plain = pd.DataFrame(wmlnb.extract(PLAIN, lambda: replay("", PLAIN), source={}))
owned = pd.DataFrame(wmlnb.extract(OWNED, lambda: replay("--owned-plain", OWNED), source={}))
pd.DataFrame(wmlnb.provenance(PLAIN, OWNED))

,extract,sha256,rows,extracted_at,commit,core,driver,mode,runs
0,notebooks/python-dialect/data/cell-replay/replay.json,6983edd81934dda6,129,2026-11-19T10:57:08+00:00,f23d0ccb5ee2,pyodide-restricted-314.0.3-35bd1d69 93eda2433fa4f650ba395f9df09f6e5c8d439844...,notebooks/python-dialect/cell-replay/replay.mjs,as recorded,58 run.json exports in ~/git/window-ml/tests/e2e/artifacts/bench/prompt-budg...
1,notebooks/python-dialect/data/cell-replay/replay-owned-plain.json,594fc4074552a1d0,129,2026-11-19T10:57:36+00:00,f23d0ccb5ee2,pyodide-restricted-314.0.3-35bd1d69 93eda2433fa4f650ba395f9df09f6e5c8d439844...,notebooks/python-dialect/cell-replay/replay.mjs,owned-plain,58 run.json exports in ~/git/window-ml/tests/e2e/artifacts/bench/prompt-budg...


## The classes, from the refusal text itself

Each refused cell carries the fork's own text (`not in dialect: ...`, the `notInDialect` a worker run puts on the card as
`askWhy`). The classes are read from that text alone. A mutation refusal names its kind ("item assignment to", "a method
that writes", "an in-place operator on" "a value the cell does not own"); the text does not say who made the value, so
the full-made / injected split is read from the counterfactual below instead.

In [2]:
CLASSES = [
    ("mutation of a value the cell does not own", r"does not own"),
    ("import", r"not in dialect: (import |a relative import)"),
    ("unlisted leaf: pd.read_csv", r"attribute 'read_csv' of module"),
    ("unlisted attribute or leaf", r"not in dialect: (attribute '|the attribute )"),
    ("unlisted call", r"not in dialect: call on "),
    ("operation on None (the cell named a table it did not load)", r"on NoneType"),
    ("construct outside the dialect", r"generator expression|While|Lambda|ClassDef"),
    ("an inside-defined function called outside", r"was defined in a cell that ran inside"),
    ("step budget, memory cap or recursion", r"step budget|memory cap|recursion past"),
    ("core without the gate", r"no membership predicate|cannot be trusted"),
    ("a refusal caught by library code", r"caught by library code"),
]

def klass(text):
    if not isinstance(text, str): return None
    for name, rx in CLASSES:
        if re.search(rx, text): return name
    return "unclassified"

for df in (plain, owned):
    df["class"] = df.refused.map(klass)
    df["outcome"] = df["class"].fillna("finished inside")
    df.loc[df.inside & df.insideError.notna(), "outcome"] = "inside, raised (no card: the error is the result)"

print(f"{len(plain)} cells in {plain.run.nunique()} runs, {plain.model.nunique()} models")
plain.outcome.value_counts().to_frame("cells").assign(share=lambda t: (t.cells / len(plain)).round(3))

129 cells in 58 runs, 6 models


,cells,share
outcome,,
unlisted leaf: pd.read_csv,41,0.318
finished inside,41,0.318
unlisted attribute or leaf,14,0.109
unlisted call,13,0.101
operation on None (the cell named a table it did not load),8,0.062
"inside, raised (no card: the error is the result)",5,0.039
construct outside the dialect,5,0.039
a refusal caught by library code,2,0.016


## By model

The share that would have run without a card (finished inside, or raised inside), per model, and each model's refusals
by class. The spread matters more than the pooled figure (caveat 3).

In [3]:
by_model = plain.groupby("model").agg(runs=("run", "nunique"), cells=("cell", "size"), no_card=("inside", "sum"))
by_model["no_card_share"] = (by_model.no_card / by_model.cells).round(2)
display(by_model.sort_values("cells", ascending=False))
pd.crosstab(plain.model, plain["class"]).loc[by_model.sort_values("cells", ascending=False).index]

,runs,cells,no_card,no_card_share
model,,,,
qwen3.6:35b,28,60,22,0.37
openrouter.minimax/minimax-m3,6,18,4,0.22
litellm.google/gemini-flash-latest,6,17,5,0.29
glm-4.7-flash:latest,6,15,6,0.40
deepseek.deepseek-flash,6,10,2,0.20
deepseek.deepseek-v4-pro,6,9,7,0.78


class,a refusal caught by library code,construct outside the dialect,operation on None (the cell named a table it did not load),unlisted attribute or leaf,unlisted call,unlisted leaf: pd.read_csv
model,,,,,,
qwen3.6:35b,1,0,7,0,2,28
openrouter.minimax/minimax-m3,0,5,1,4,0,4
litellm.google/gemini-flash-latest,0,0,0,4,8,0
glm-4.7-flash:latest,1,0,0,1,0,7
deepseek.deepseek-flash,0,0,0,4,3,1
deepseek.deepseek-v4-pro,0,0,0,1,0,1


## Cards per run

A card is a refused cell (its full-path rerun is what the person approves). Recorded: every cell asked, so the recorded
cards per run are the cells per run. Replayed: only the refused ones.

In [4]:
per_run = plain.groupby(["model", "run"]).agg(cells=("cell", "size"), cards=("inside", lambda s: int((~s).sum()))).reset_index()
print("cards per run, replayed:", per_run.cards.value_counts().sort_index().to_dict())
print("runs with no card left:", int((per_run.cards == 0).sum()), "of", len(per_run))
per_run.groupby("model").agg(runs=("run", "size"), cards_recorded=("cells", "sum"), cards_replayed=("cards", "sum"),
                             mean_replayed=("cards", "mean")).round(2)

cards per run, replayed: {0: 8, 1: 29, 2: 15, 3: 2, 4: 2, 5: 2}
runs with no card left: 8 of 58


,runs,cards_recorded,cards_replayed,mean_replayed
model,,,,
deepseek.deepseek-flash,6,10,8,1.33
deepseek.deepseek-v4-pro,6,9,2,0.33
glm-4.7-flash:latest,6,15,9,1.50
litellm.google/gemini-flash-latest,6,17,12,2.00
openrouter.minimax/minimax-m3,6,18,14,2.33
qwen3.6:35b,28,60,38,1.36


## The counterfactual: mutation of plain data permitted

Simulated without a fork change: before each cell, every plain-data value in the namespace that the cell did not make (a
DataFrame, Series, ndarray, list, dict, set, str or number: the injected `df`/`tables` and whatever a full-path cell left)
is rebuilt by `copy.deepcopy` inside an inside cell, so it is owned. The change removes a card only where a cell's
refusal is a mutation refusal; the upper bound is the count of cells whose FIRST refusal is that class, since a later
refusal in the same cell is hidden behind it.

In [5]:
j = plain[["run", "cell", "model", "outcome"]].merge(owned[["run", "cell", "outcome"]], on=["run", "cell"], suffixes=("", "_owned"))
moved = j[j.outcome != j.outcome_owned]
print("cells whose first refusal is a mutation refusal (upper bound on what the change removes):",
      int((plain["class"] == "mutation of a value the cell does not own").sum()))
print("cells whose outcome changed with plain data owned:", len(moved))
moved

cells whose first refusal is a mutation refusal (upper bound on what the change removes): 0
cells whose outcome changed with plain data owned: 0


,run,cell,model,outcome,outcome_owned


## Behind the first refusal: a syntactic count of the stores

The bound above is on the mutation change ALONE. A refused cell stops at its first refusal, and nothing after that line
was judged: 83 cells stop at a refusal of another class, 41 of them at `pd.read_csv`. If the vocabulary is widened first,
which is what the counts above point to, those cells run further, and a mutation refusal can appear behind the first one.

So this counts, from the recorded source by `ast` and without running it, the stores the gate would judge as a mutation
of a value the cell does not own: item or attribute assignment, an augmented assignment, `del x[...]`, a call with
`inplace=True`, or a listed mutating method of list, dict or set, each on a ROOT name the cell did not bind before that
line (an import binds a module the cell did not make, so it does not count). SYNTACTIC: a store in a branch that never
runs counts, and one made through an alias the cell bound (`y = df; y["a"] = 0`) is missed. The extract
(`stores.json`, from `cell-replay/stores.py`) holds counts per kind, never the code.

In [6]:
STORES = HERE / "data" / "cell-replay" / "stores.json"
st = pd.DataFrame(wmlnb.extract(STORES, lambda: replay("stores", STORES), source={}))
KINDS = ["item", "attr", "aug", "aug_name", "del", "inplace", "method"]
st["parsed"] = st.stores.notna()
kinds = pd.DataFrame([d if isinstance(d, dict) else {k: 0 for k in KINDS} for d in st.stores])[KINDS]
st = pd.concat([st.drop(columns="stores"), kinds], axis=1)
st["any_store"] = st[KINDS].sum(axis=1) > 0
refused = plain[plain.refused.notna()].merge(st, on=["run", "cell", "model"])
print(f"{len(refused)} refused cells; {int((~refused.parsed).sum())} do not parse as Python")
print("refused cells with a store on a value the cell did not make:", int(refused.any_store.sum()))
display(refused.groupby("class")[["any_store"]].sum().assign(cells=refused.groupby("class").size()))
refused.loc[refused.any_store, ["model", "class"] + KINDS]

83 refused cells; 0 do not parse as Python
refused cells with a store on a value the cell did not make: 1


,any_store,cells
class,,
a refusal caught by library code,0,2
construct outside the dialect,0,5
operation on None (the cell named a table it did not load),0,8
unlisted attribute or leaf,0,14
unlisted call,1,13
unlisted leaf: pd.read_csv,0,41


,model,class,item,attr,aug,aug_name,del,inplace,method
15,qwen3.6:35b,unlisted call,0,1,0,0,0,0,0


The one store is an attribute assignment on `decimal`'s context (`getcontext().prec = ...`). That is library state, not
plain data that a full-path cell made or that the harness injected, so a change that permits mutating such data would not
admit it either. On this sample, then, no refused cell has a store on plain data it did not make, even behind its first
refusal (syntactically, with the caveats above).

## Two notes on single classes

- `attribute 'columns' of DataFrame` and `attribute 'shape' of DataFrame` (one cell each): nothing differs in those
  cells. Neither attribute is in `manifest-pandas.json` (`columns` appears there only as the dynamic rule for a
  column read by name, `df.amount`), and every form of the read is refused on the injected frame, `df.shape`,
  `x = df.shape`, `print(df.columns)` and `list(df.columns)` alike (probed through `replay.mjs` on synthetic cells).
  No recorded cell reads either inside: the three others that read them stopped at an earlier refusal.
- The "operation on None" cells are not the replay's artefact. Their recorded results are the same `TypeError` on `None`:
  the run loaded its table under another name, so `df` was `None` in the real run as well (counted below, from the
  recorded result text).

In [7]:
none_cells = plain[plain["class"] == "operation on None (the cell named a table it did not load)"].merge(st[["run", "cell", "recorded_none"]], on=["run", "cell"])
print(f"{len(none_cells)} replayed 'operation on None' cells, {int(none_cells.recorded_none.sum())} with the TypeError on None in the recorded result; "
      f"{int(st.recorded_none.sum())} recorded cells in all")

8 replayed 'operation on None' cells, 8 with the TypeError on None in the recorded result; 9 recorded cells in all


## The unclassified remainder, with examples

A refusal no class above matched, with its text (line numbers stripped). Empty means every refusal was classified.

In [8]:
rest = plain[plain["class"] == "unclassified"]
rest.refused.str.replace(r"\(line \d+\)", "(line N)", regex=True).value_counts().to_frame("cells")

,cells
refused,


## Every refusal text, counted

What the classes were read from, line numbers stripped.

In [9]:
plain.refused.dropna().str.replace(r"\(line \d+\)", "(line N)", regex=True).str.replace("not in dialect: ", "").value_counts().to_frame("cells")

,cells
refused,
attribute 'read_csv' of module (line N),41
call on type (line N),11
item access on NoneType (line N),8
a generator expression outside consumer position (line N),5
attribute 'keys' of FrameLocalsProxy (line N),3
attribute 'iloc' of Series (line N),3
attribute 'rint' of module (line N),3
a refusal during the run was caught by library code,2
call on builtin_function_or_method (line N),2
